In [29]:
import pandas as pd
dftrain = pd.read_csv("../train_data_2.csv")
dftest = pd.read_csv("../test_data_2.csv")
# Trouver les colonnes communes
common_cols = dftrain.columns.intersection(dftest.columns)

# Garder uniquement les colonnes communes
dftrain = dftrain[common_cols].dropna()
dftest = dftest[common_cols].dropna()

# Sélection des features et labels
features = [
    "num_nodes", "num_edges", "avg_degree", "num_scc", "avg_clustering", 
    "avg_degree_centrality", "avg_closeness_centrality", "avg_betweenness_centrality", 
    "num_cycles", "avg_cycle_length"
]

#X = df[features]  # Features
X = dftrain.iloc[:, 454:460]
X1 = dftrain.iloc[:, 460:].applymap(lambda x: 1 if x > 0 else 0)
X = pd.concat([X, X1], axis=1)

y = dftrain.iloc[:, 1:454]  # Labels binaires/multiclasses

/var/folders/hv/y00c8cl91z34xlwn9xbh5cfr0000gn/T/ipykernel_65153/1046001878.py:20: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  X1 = dftrain.iloc[:, 460:].applymap(lambda x: 1 if x > 0 else 0)


In [30]:
X

,num_nodes,num_edges,avg_degree,density,num_scc,avg_clustering,scasd,iretd,push,retf,...,pminuw,vpaddusw,phaddw,pminud,pblendvb,vcmpps,rcpss,cmpeqpd,vpsubusw,cmpnltsd
0,10,6,1.200000,0.066667,10,0.000000,1,1,1,1,...,0,0,0,0,0,0,0,0,0,0
1,143,110,1.538462,0.005417,143,0.000000,0,0,1,0,...,0,0,0,0,0,0,0,0,0,0
2,213,159,1.492958,0.003521,207,0.005477,0,0,1,0,...,0,0,0,0,0,0,0,0,0,0
3,944,794,1.682203,0.000892,921,0.032920,0,0,1,0,...,0,0,0,0,0,0,0,0,0,0
4,103,81,1.572816,0.007710,103,0.000000,0,0,1,0,...,0,0,0,0,0,0,0,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
23095,7099,5061,1.425835,0.000100,7086,0.003904,0,0,1,0,...,0,0,0,0,0,0,0,0,0,0
23096,13051,10999,1.685541,0.000065,12970,0.024440,0,1,1,0,...,0,0,0,0,0,0,0,0,0,0
23097,18314,14439,1.576826,0.000043,18304,0.000446,0,0,1,0,...,0,0,0,0,0,0,0,0,0,0
23098,15858,13246,1.670576,0.000053,15547,0.033780,0,0,1,0,...,0,0,0,0,0,0,0,0,0,0


In [25]:
from sklearn.metrics import f1_score
from sklearn.model_selection import train_test_split
from sklearn.neighbors import KNeighborsClassifier
from sklearn.multioutput import MultiOutputClassifier

# Division en train (80%) et test (20%)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
scale_pos_weights = (y_train.shape[0] - y_train.sum(axis=0)) / (y_train.sum(axis=0) + 1e-6)
# Modèle KNN avec k=5
knn = MultiOutputClassifier(KNeighborsClassifier(n_neighbors=5))
knn.fit(X_train, y_train)
y_pred = knn.predict(X_test)

# Calcul du F1-score pour chaque label (chaque sortie)
f1_macro_per_label = []
for i in range(y_test.shape[1]):
    f1_macro_per_label.append(f1_score(y_test.iloc[:, i], y_pred[:, i], average="weighted", zero_division=0))

# Calcul du F1-score macro moyen
f1_macro = sum(f1_macro_per_label) / 453
print(f"F1-score macro: {f1_macro:.4f}")

F1-score macro: 0.9357


In [28]:
f1_macro = f1_score(y_test, y_pred, average='macro', zero_division=0)
f1_macro

0.020043174281871915

In [ ]:

# Lecture du fichier de test
df_to_predict = pd.read_csv("../test_data_1.csv")
df_to_predict
names = df_to_predict["name"]

# Supprimer la colonne "name" pour conserver uniquement les features
df_features = df_to_predict.drop("name", axis=1)

# Assurez-vous que les features correspondent à celles utilisées lors de l'entraînement
X_to_predict = df_features[X.columns]

# Effectuer la prédiction avec le modèle multi-output
predictions = knn.predict(X_to_predict)

# Créer un DataFrame pour les prédictions avec les mêmes noms de colonnes que pour y_train
predictions_df = pd.DataFrame(predictions, columns=y_train.columns)

# Réinsérer la colonne "name" en première position
predictions_df.insert(0, "name", names.values)

# Exporter les prédictions dans un fichier CSV
predictions_df.to_csv("predictions_knn.csv", index=False)
predictions_df